# ATML PA3 - Task 1 runner (Kaggle, GPU T4 x2, Internet on)

All logic lives in the repo; this notebook only clones it, restores earlier outputs attached as inputs, and runs the job queue.

In [ ]:
%cd /kaggle/working
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/atml-pa3 && git clone -q https://github.com/sayim665/atml-pa3.git /kaggle/working/atml-pa3
!cd /kaggle/working/atml-pa3 && git log -1 --oneline
!pip install -q "lm-eval>=0.4.8" 2>&1 | tail -n 2
!python -c "import torch, transformers, lm_eval; print(torch.__version__, transformers.__version__)"

In [ ]:
import glob, os, shutil
os.environ["ATML_OUT"] = "/kaggle/working/task1_outputs"
os.environ["ATML_RESULTS"] = "/kaggle/working/task1_results"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for name in ("task1_outputs", "task1_results"):
    for previous in sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)):
        print("restoring", previous)
        shutil.copytree(previous, f"/kaggle/working/{name}", dirs_exist_ok=True)
os.chdir("/kaggle/working/atml-pa3")

In [ ]:
!python -m task1.data

In [ ]:
JOBS = "train:full_ft train:l2sp train:lora16 train:oplora16 train:lora4 eval:baseline eval:full_ft_mid eval:full_ft eval:l2sp eval:lora16 eval:oplora16 eval:lora4"
!python -m task1.run_queue --jobs {JOBS} --budget-hours 11.3

In [ ]:
!cat /kaggle/working/task1_outputs/queue_status.json
!for f in /kaggle/working/task1_outputs/logs/*.log; do echo "== $f"; tail -n 3 "$f"; done
!du -sh /kaggle/working/task1_outputs /kaggle/working/task1_results